# 복무 만족도 예측
시작: 2026-10-09 14:45 KST · 종료: 16:35 KST
문제·채점 기준은 README.md를 확인하세요. 아래 기본 예제를 개선하고 submission.csv를 저장하세요.
서버 전체에서 커널 하나만 사용합니다. 완료 후 Kernel → Shutdown으로 실행 자리를 반납하세요.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
DATA=Path('data')
train=pd.read_csv(DATA/'train.csv'); test=pd.read_csv(DATA/'test.csv'); sample=pd.read_csv(DATA/'sample_submission.csv')
print(train.shape, test.shape)


(80, 6) (20, 5)


## 풀이: 선형 회귀와 '4항목 평균' 선형 회귀의 평균
- **데이터 구조**: 네 항목 점수의 선형 결합에 잡음이 더해진 형태입니다(훈련 R2 0.83, 잔차 표준편차 0.44). 계수가 0.33/0.27/0.25/0.24 로 비슷하고 입력끼리 상관이 거의 없습니다.
- **모델**: ① 계수 4개를 자유롭게 두는 선형 회귀와 ② 4항목 평균 하나만 쓰는 선형 회귀(같은 가중치로 수축)의 예측을 평균합니다. 80행으로 계수 4개를 추정하는 불안정을 줄이는 목적입니다.
- **검증**(5-Fold x 20회 반복 RMSE): 제공 베이스라인 RF 0.696, 선형 0.4628, 평균 선형 0.4647, **두 모델 평균 0.4585**. 선형 대비 차이는 0.004 로 작고(20회 중 17회에서 낮음), 잔차 표준편차 0.44 가 사실상 하한이라 더 줄이기 어렵습니다.
- 기각: Ridge(0.465~0.484), 2차 다항(0.466), HGB(0.649), 반올림 후처리(0.464).

In [2]:
from sklearn.model_selection import KFold
R = 20   # 5-Fold 를 20번 반복 (학습 80행뿐이라 단일 분할 점수는 변동이 큼)
def cv_rmse(X, y, make, post=None):
    out = []
    for r in range(R):
        oof = np.zeros(len(y))
        for a, b in KFold(5, shuffle=True, random_state=r).split(X):
            p = make().fit(X.iloc[a], y.iloc[a]).predict(X.iloc[b]); oof[b] = post(p) if post else p
        out.append(np.sqrt(np.mean((y.values - oof) ** 2)))
    return np.array(out)
from sklearn.linear_model import LinearRegression

class MeanLin:
    """4항목 평균 한 변수로 선형 회귀"""
    def fit(self, X, y): self.m = LinearRegression().fit(X.mean(axis=1).values.reshape(-1, 1), y); return self
    def predict(self, X): return self.m.predict(X.mean(axis=1).values.reshape(-1, 1))
class Blend:
    def fit(self, X, y): self.a = LinearRegression().fit(X, y); self.b = MeanLin().fit(X, y); return self
    def predict(self, X): return (self.a.predict(X) + self.b.predict(X)) / 2

features = [c for c in test.columns if c != "soldier_id"]; X, y = train[features], train["overall_satisfaction"]
a = cv_rmse(X, y, LinearRegression); b = cv_rmse(X, y, MeanLin); c = cv_rmse(X, y, Blend)
for n, v in (("선형", a), ("평균 한 변수 선형", b), ("두 모델 평균 (메인)", c)):
    print(f"{n:22s} RMSE {v.mean():.4f} (반복 sd {v.std():.4f}) -> 점수 {100 / (1 + v.mean() / 1.05889019):.2f}")
print("메인이 선형보다 낮은 반복:", int((c < a).sum()), "/", R)

model = Blend().fit(X, y)
print("선형 계수:", dict(zip(features, model.a.coef_.round(3))), "절편 %.3f" % model.a.intercept_)
submission = pd.DataFrame({"soldier_id": test.soldier_id, "overall_satisfaction": model.predict(test[features])})[sample.columns]
assert np.isfinite(submission.overall_satisfaction).all()
submission.to_csv("submission.csv", index=False)
print("예측 범위 %.2f ~ %.2f (train %.1f ~ %.1f)" % (submission.overall_satisfaction.min(), submission.overall_satisfaction.max(), y.min(), y.max())); submission.head()

선형                     RMSE 0.4628 (반복 sd 0.0082) -> 점수 69.59
평균 한 변수 선형             RMSE 0.4647 (반복 sd 0.0064) -> 점수 69.50
두 모델 평균 (메인)           RMSE 0.4585 (반복 sd 0.0066) -> 점수 69.78
메인이 선형보다 낮은 반복: 17 / 20
선형 계수: {'leadership_score': np.float64(0.333), 'facility_score': np.float64(0.268), 'meal_score': np.float64(0.248), 'rest_time_score': np.float64(0.235)} 절편 -0.611
예측 범위 4.62 ~ 8.02 (train 4.1 ~ 8.4)


,soldier_id,overall_satisfaction
0,S001,5.965772
1,S005,6.509870
2,S011,6.427688
3,S013,5.151778
4,S019,6.354355


In [3]:
check = pd.read_csv('submission.csv')
assert check.columns.tolist() == sample.columns.tolist()
assert check.iloc[:,0].tolist() == sample.iloc[:,0].tolist()
assert check.iloc[:,0].is_unique and not check.isna().any().any()
print('제출 형식 확인:', check.shape)


제출 형식 확인: (20, 2)
